In [ ]:
import os
import torch
import matplotlib.pyplot as plt

from diffusers import StableDiffusion3Pipeline, FlowMatchEulerDiscreteScheduler
from diffusers.schedulers.scheduling_flow_match_euler_discrete import (
    FlowMatchEulerDiscreteSchedulerOutput,
)
from diffusers.utils.torch_utils import randn_tensor


class OvershootNoiseFlowMatchEulerScheduler(FlowMatchEulerDiscreteScheduler):
    def __init__(
        self,
        *args,
        overshoot_scale=1.0,
        overshoot_start_ratio=0.25,
        overshoot_end_ratio=0.60,
        noise_scale=0.0,
        noise_start_ratio=0.1,
        noise_end_ratio=0.4,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        self.overshoot_scale = overshoot_scale
        self.overshoot_start_ratio = overshoot_start_ratio
        self.overshoot_end_ratio = overshoot_end_ratio
        self.noise_scale = noise_scale
        self.noise_start_ratio = noise_start_ratio
        self.noise_end_ratio = noise_end_ratio

    def _is_in_ratio_range(self, start_ratio, end_ratio):
        if self.num_inference_steps is None or self.step_index is None:
            return False

        start = int(start_ratio * self.num_inference_steps)
        end = int(end_ratio * self.num_inference_steps)

        return start <= self.step_index <= end

    def step(
        self,
        model_output,
        timestep,
        sample,
        s_churn=0.0,
        s_tmin=0.0,
        s_tmax=float("inf"),
        s_noise=1.0,
        generator=None,
        per_token_timesteps=None,
        return_dict=True,
    ):
        if self.step_index is None:
            self._init_step_index(timestep)

        orig_dtype = sample.dtype
        device = sample.device

        sigma_idx = self.step_index
        sigma = self.sigmas[sigma_idx].to(device=device, dtype=orig_dtype)
        sigma_next = self.sigmas[sigma_idx + 1].to(device=device, dtype=orig_dtype)

        dt = sigma_next - sigma

        overshoot_factor = 1.0
        if self._is_in_ratio_range(
            self.overshoot_start_ratio,
            self.overshoot_end_ratio,
        ):
            overshoot_factor = self.overshoot_scale

        prev_sample = sample + overshoot_factor * dt * model_output

        if (
            self.noise_scale > 0
            and self._is_in_ratio_range(
                self.noise_start_ratio,
                self.noise_end_ratio,
            )
        ):
            noise = randn_tensor(
                prev_sample.shape,
                generator=generator,
                device=prev_sample.device,
                dtype=orig_dtype,
            )

            step_noise_scale = self.noise_scale * torch.abs(dt)
            prev_sample = prev_sample + step_noise_scale * noise

        self._step_index += 1
        prev_sample = prev_sample.to(orig_dtype)

        if not return_dict:
            return (prev_sample,)

        return FlowMatchEulerDiscreteSchedulerOutput(prev_sample=prev_sample)


def main():
    model_id = "stabilityai/stable-diffusion-3.5-medium"
    out_dir = "./sd3_sampler_ablation"
    os.makedirs(out_dir, exist_ok=True)

    device = "cuda"
    dtype = torch.float16

    pipe = StableDiffusion3Pipeline.from_pretrained(
        model_id,
        torch_dtype=dtype,
    )
    pipe = pipe.to(device)

    # 메모리 부족 시 사용
    # pipe.enable_model_cpu_offload()
    # pipe.enable_attention_slicing()

    prompt = "a dog without teeth, realistic photo, close-up face"

    negative_prompt = (
        "teeth, visible teeth, open mouth, cartoon, painting, "
        "blurry, low quality, deformed"
    )

    num_inference_steps = 28
    guidance_scale = 7
    num_seeds = 8

    configs = [
        {
            "name": "base",
            "overshoot_scale": 1.00,
            "noise_scale": 0.00,
        },
        {
            "name": "over_150",
            "overshoot_scale": 1.50,
            "noise_scale": 0.00,
        },
        {
            "name": "over_200",
            "overshoot_scale": 2.00,
            "noise_scale": 0.00,
        },
        {
            "name": "over_200_noise_100",
            "overshoot_scale": 2.00,
            "noise_scale": 1.00,
        },
        
        {
            "name": "over_200_noise_200",
            "overshoot_scale": 2.00,
            "noise_scale": 2.00,
        },
    ]

    all_images = {}
    base_config = pipe.scheduler.config

    for cfg in configs:
        print(f"\n[Config] {cfg['name']}")

        pipe.scheduler = OvershootNoiseFlowMatchEulerScheduler.from_config(
            base_config,
            overshoot_scale=cfg["overshoot_scale"],
            overshoot_start_ratio=0.25,
            overshoot_end_ratio=0.60,
            noise_scale=cfg["noise_scale"],
            noise_start_ratio=0.35,
            noise_end_ratio=0.65,
        )

        for seed in range(num_seeds):
            print(f"  seed {seed}")

            generator = torch.Generator(device=device).manual_seed(seed)

            with torch.no_grad():
                image = pipe(
                    prompt=prompt,
                    negative_prompt=negative_prompt,
                    num_inference_steps=num_inference_steps,
                    guidance_scale=guidance_scale,
                    generator=generator,
                ).images[0]

            if seed not in all_images:
                all_images[seed] = {}

            all_images[seed][cfg["name"]] = image

    # ============================================================
    # Grid 저장
    # row = seed
    # col = config
    # ============================================================

    num_configs = len(configs)

    fig, axes = plt.subplots(
        num_seeds,
        num_configs,
        figsize=(4 * num_configs, 4 * num_seeds),
    )

    if num_seeds == 1:
        axes = axes[None, :]
    if num_configs == 1:
        axes = axes[:, None]

    for seed in range(num_seeds):
        for col, cfg in enumerate(configs):
            ax = axes[seed, col]
            img = all_images[seed][cfg["name"]]

            ax.imshow(img)
            ax.axis("off")

            if seed == 0:
                ax.set_title(cfg["name"], fontsize=12)

            if col == 0:
                ax.set_ylabel(f"seed {seed}", fontsize=12)

    plt.tight_layout()

    grid_save_path = os.path.join(out_dir, "sd3_sampler_ablation_grid.png")
    plt.savefig(grid_save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    print(f"\nDone. Grid saved to: {grid_save_path}")


if __name__ == "__main__":
    main()

Loading pipeline components...: 100%|██████████| 9/9 [00:08<00:00,  1.01it/s]



[Config] base
  seed 0


100%|██████████| 28/28 [00:02<00:00,  9.33it/s]


  seed 1


100%|██████████| 28/28 [00:02<00:00,  9.34it/s]


...

  seed 6


100%|██████████| 28/28 [00:02<00:00,  9.36it/s]


  seed 7


100%|██████████| 28/28 [00:02<00:00,  9.37it/s]



Done. Grid saved to: ./sd3_sampler_ablation/sd3_sampler_ablation_grid.png
